# Team [42] AI´m lost

### Short Notebook 1

#### Elias Strømsnes - studnummer

#### Hannah Lervik - 590345

#### Kristian Nyland Larsen - 590348

# Imports

In [1]:
import os
# Set environment variables BEFORE heavy libs import for determinism
os.environ["PYTHONHASHSEED"] = "42"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import pandas as pd
from prophet import Prophet
from tqdm import tqdm

import logging
logging.getLogger('prophet').setLevel(logging.WARNING)


# Load data
### We landed on only using receivals data

In [2]:
receivals = pd.read_csv('data/kernel/receivals.csv', parse_dates=['date_arrival'])
receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'], utc=True).dt.tz_localize(None)

receivals = receivals[receivals['receival_status'] == 'Completed'].copy()
receivals = receivals[receivals['net_weight'] >= 0].copy()

receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'])

daily_aggregated = (
    receivals.groupby(['rm_id', pd.Grouper(key='date_arrival', freq='D')])['net_weight']
    .sum().reset_index()
)
print(daily_aggregated)

        rm_id date_arrival  net_weight
0       342.0   2004-06-23     24940.0
1       343.0   2005-03-29     21760.0
2       345.0   2004-09-01     22780.0
3       346.0   2004-06-24       820.0
4       346.0   2004-06-30     21260.0
...       ...          ...         ...
41900  4463.0   2024-10-17      2000.0
41901  4481.0   2024-10-29     24680.0
41902  4481.0   2024-11-27     22340.0
41903  4501.0   2024-12-02     23580.0
41904  4501.0   2024-12-09     24600.0

[41905 rows x 3 columns]


# Prophet Prediction Functions

In [3]:
def create_material_predictions(material_data, material_id=None):
    prophet_df = _prepare_prophet_dataframe(material_data)
    complete_df = _fill_missing_dates(prophet_df)
    enriched_df = _add_time_features(complete_df)
    trained_model = _setup_and_train_prophet(enriched_df)
    predictions = _generate_future_predictions(trained_model)
    return predictions


def _prepare_prophet_dataframe(raw_data):
    return raw_data.copy().rename(columns={
        'date_arrival': 'ds',
        'net_weight': 'y'
    })


def _fill_missing_dates(prophet_data):
    start_date = prophet_data["ds"].min().normalize()
    end_date = prophet_data["ds"].max().normalize()
    complete_date_range = pd.date_range(start_date, end_date, freq="D")
    date_skeleton = pd.DataFrame({"ds": complete_date_range})
    return date_skeleton.merge(prophet_data, on="ds", how="left").fillna({"y": 0})


def _add_time_features(base_df):
    enhanced_df = base_df.copy()
    enhanced_df["is_weekday"] = (enhanced_df["ds"].dt.dayofweek < 5).astype(int)
    return enhanced_df


def _setup_and_train_prophet(training_data):
    # Ensure deterministic ordering
    training_data = training_data.sort_values('ds').reset_index(drop=True)

    # Prophet init (ingen 'seed' param i denne versjonen). mcmc_samples=0 gir deterministisk fit.
    prophet_model = Prophet(
        growth="linear",
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="multiplicative",
        changepoint_prior_scale=0.02,
        changepoint_range=0.8,
        seasonality_prior_scale=0.69,
        holidays_prior_scale=0.56,
        mcmc_samples=0,  # disable stochastic sampling
    )

    prophet_model.add_country_holidays(country_name="NO")
    prophet_model.add_regressor("is_weekday")
    prophet_model.fit(training_data)
    return prophet_model


def _generate_future_predictions(model):
    future_dates = pd.DataFrame({
        "ds": pd.date_range("2025-01-01", "2025-05-31", freq="D")
    })
    future_dates["is_weekday"] = (future_dates["ds"].dt.dayofweek < 5).astype(int)
    raw_forecast = model.predict(future_dates)
    processed_forecast = _post_process_predictions(raw_forecast)
    return processed_forecast


def _post_process_predictions(raw_predictions):
    final_predictions = raw_predictions[["ds", "yhat"]].copy()
    final_predictions["yhat_scaled"] = final_predictions["yhat"] * 0.62
    final_predictions["yhat_scaled"] = final_predictions["yhat_scaled"].clip(lower=0)
    return final_predictions[["ds", "yhat_scaled"]]


# Process All Materials

In [4]:
def process_material_forecasting(receivals_data, aggregated_data, min_observations=20):
    all_materials = sorted(receivals_data["rm_id"].unique())
    
    valid_materials, excluded_materials = _filter_materials_by_data_quality(
        all_materials, aggregated_data, min_observations
    )
    
    material_forecasts = _batch_generate_predictions(valid_materials, aggregated_data)
    
    consolidated_predictions = _consolidate_prediction_results(material_forecasts)
    
    _report_processing_summary(excluded_materials)
    
    return consolidated_predictions


def _filter_materials_by_data_quality(material_ids, data, threshold):
    valid_ids = []
    excluded_ids = []
    
    for material_id in tqdm(material_ids, desc="Evaluerer datakvalitet"):
        material_subset = data[data["rm_id"] == material_id]
        
        if len(material_subset) >= threshold:
            valid_ids.append(material_id)
        else:
            excluded_ids.append(material_id)
    
    return valid_ids, excluded_ids


def _batch_generate_predictions(material_list, source_data):
    prediction_collection = []
    
    for current_material in tqdm(material_list, desc="Genererer prediksjoner"):
        material_timeseries = source_data[source_data["rm_id"] == current_material]
        
        forecast_result = create_material_predictions(material_timeseries, material_id=current_material)
        
        forecast_result = _attach_material_id(forecast_result, current_material)
        
        prediction_collection.append(forecast_result)
    
    return prediction_collection


def _attach_material_id(prediction_df, material_id):
    enhanced_prediction = prediction_df.copy()
    enhanced_prediction["rm_id"] = material_id
    return enhanced_prediction


def _consolidate_prediction_results(prediction_list):
    return pd.concat(prediction_list, ignore_index=True)


def _report_processing_summary(excluded_list):
    print(f"\nHoppet over {excluded_list} materialer med for lite data.")

pred2025 = process_material_forecasting(receivals, daily_aggregated)

Genererer prediksjoner:   0%|          | 0/96 [00:00<?, ?it/s]17:28:17 - cmdstanpy - INFO - Chain [1] start processing
17:28:17 - cmdstanpy - INFO - Chain [1] start processing
17:28:17 - cmdstanpy - INFO - Chain [1] done processing
17:28:17 - cmdstanpy - INFO - Chain [1] done processing
Genererer prediksjoner:   1%|          | 1/96 [00:00<00:33,  2.81it/s]17:28:18 - cmdstanpy - INFO - Chain [1] start processing
17:28:18 - cmdstanpy - INFO - Chain [1] start processing
17:28:18 - cmdstanpy - INFO - Chain [1] done processing
17:28:18 - cmdstanpy - INFO - Chain [1] done processing
17:28:18 - cmdstanpy - INFO - Chain [1] start processing
17:28:18 - cmdstanpy - INFO - Chain [1] start processing
17:28:18 - cmdstanpy - INFO - Chain [1] done processing
17:28:18 - cmdstanpy - INFO - Chain [1] done processing
Genererer prediksjoner:   3%|▎         | 3/96 [00:00<00:14,  6.38it/s]17:28:18 - cmdstanpy - INFO - Chain [1] start processing
17:28:18 - cmdstanpy - INFO - Chain [1] start processing
17:28:


Hoppet over [342.0, 343.0, 345.0, 346.0, 347.0, 348.0, 353.0, 354.0, 355.0, 358.0, 360.0, 362.0, 374.0, 378.0, 380.0, 381.0, 383.0, 387.0, 388.0, 390.0, 1842.0, 1843.0, 1844.0, 1845.0, 1846.0, 1850.0, 1851.0, 1852.0, 1853.0, 1854.0, 1857.0, 1858.0, 1866.0, 1867.0, 1868.0, 1872.0, 1874.0, 1882.0, 1981.0, 2001.0, 2061.0, 2102.0, 2121.0, 2122.0, 2128.0, 2141.0, 2148.0, 2156.0, 2158.0, 2201.0, 2223.0, 2282.0, 2283.0, 2285.0, 2302.0, 2322.0, 2323.0, 2341.0, 2343.0, 2344.0, 2345.0, 2347.0, 2348.0, 2362.0, 2363.0, 2421.0, 2561.0, 2742.0, 2821.0, 2841.0, 2861.0, 3022.0, 3101.0, 3144.0, 3201.0, 3222.0, 3381.0, 3461.0, 3481.0, 3501.0, 3541.0, 3581.0, 3601.0, 3621.0, 3701.0, 3762.0, 3802.0, 3821.0, 3921.0, 3941.0, 4021.0, 4044.0, 4081.0, 4101.0, 4161.0, 4263.0, 4302.0, 4343.0, 4381.0, 4401.0, 4441.0, 4443.0, 4461.0, 4462.0, 4463.0, 4481.0, 4501.0] materialer med for lite data.


# Apply Activity Filter

In [5]:
def apply_recent_activity_filter(prediction_data, historical_data, lookback_days=90):
    
    analysis_period = _define_analysis_timeframe(lookback_days)
    
    recently_active_materials = _identify_active_materials(historical_data, analysis_period)
    
    filtered_predictions = _zero_out_inactive_predictions(prediction_data, recently_active_materials)
    
    return filtered_predictions


def _define_analysis_timeframe(days_back):
    reference_date = pd.Timestamp('2024-12-31')
    lookback_start = reference_date - pd.Timedelta(days=days_back-1)  #-1 to include reference date
    
    return {
        'start_date': lookback_start,
        'end_date': reference_date
    }


def _identify_active_materials(data_source, time_window):
    
    activity_criteria = (
        (data_source['date_arrival'] >= time_window['start_date']) & 
        (data_source['date_arrival'] <= time_window['end_date']) & 
        (data_source['net_weight'] > 0)
    )
    
    active_material_set = set(sorted(data_source.loc[activity_criteria, 'rm_id'].unique()))
    
    return active_material_set


def _zero_out_inactive_predictions(predictions, active_material_ids):

    inactive_materials_mask = ~predictions['rm_id'].isin(active_material_ids)
    
    updated_predictions = predictions.copy()
    updated_predictions.loc[inactive_materials_mask, 'yhat_scaled'] = 0.0
    
    return updated_predictions

pred2025 = apply_recent_activity_filter(pred2025, daily_aggregated)

# Transform to Cumulative Format

In [6]:
def transform_to_cumulative_format(predictions_df):
    
    formatted_data = _standardize_column_names(predictions_df)
    
    enriched_data = _compute_cumulative_weights(formatted_data)
    
    return enriched_data

def _standardize_column_names(data):
    return data.rename(columns={
        "ds": "date", 
        "yhat_scaled": "pred_net_weight"
    })

def _compute_cumulative_weights(data):
    processed_data = data.copy()

    processed_data = processed_data.sort_values(["rm_id", "date"])
    
    processed_data["cum_weight"] = (
        processed_data.groupby("rm_id")["pred_net_weight"].cumsum()
    )
    
    return processed_data

pred2025cum = transform_to_cumulative_format(pred2025)

# Create Final Submission

In [7]:
def create_final_submission(predictions_data, mapping_file="data/prediction_mapping.csv", output_file="submission.csv"):
    clean_predictions, clean_mapping = _prepare_datasets_for_merge(predictions_data, mapping_file)

    combined_data = _merge_predictions_with_mapping(clean_predictions, clean_mapping)
    
    final_submission = _format_submission_output(combined_data)
    
    _save_submission_file(final_submission, output_file)
    
    return final_submission


def _prepare_datasets_for_merge(predictions, mapping):
    mapping_data = pd.read_csv(mapping)

    clean_pred = predictions.copy()
    clean_pred["date"] = pd.to_datetime(clean_pred["date"])
    clean_pred["rm_id"] = clean_pred["rm_id"].astype(float)
    
    clean_map = mapping_data.copy()
    clean_map["forecast_end_date"] = pd.to_datetime(clean_map["forecast_end_date"], errors="coerce")
    clean_map["rm_id"] = clean_map["rm_id"].astype(float)
    
    return clean_pred, clean_map


def _merge_predictions_with_mapping(predictions, mapping):
    merged_result = mapping.merge(
        predictions,
        left_on=["rm_id", "forecast_end_date"],
        right_on=["rm_id", "date"],
        how="left"
    )
    
    merged_result["cum_weight"] = merged_result["cum_weight"].fillna(0)
    
    return merged_result


def _format_submission_output(merged_data):
    submission_data = merged_data[["ID", "cum_weight"]].copy()
    submission_data = submission_data.rename(columns={"cum_weight": "predicted_weight"})
    
    return submission_data

def _save_submission_file(data, filename):
    data.to_csv(filename, index=False)
    

submission = create_final_submission(pred2025cum)